# Calculating Confidence Intervals: the Recipe and a Worked Example

Everything from the last three notebooks compresses into a six-step recipe: sample mean → confidence level → critical value → standard error → margin of error → interval. Run it once on real numbers — 49 of Statistopia's 6,000 adults, $\bar{x} = 170$ cm, $\sigma = 25$ cm, 95% — and the margin comes out to a conveniently round $1.96 \times 25/\sqrt{49} = 7$ cm, giving the interval $[163, 177]$. Plus the fine print: the recipe assumes a **random sample** and either $n > 30$ or an approximately normal population. It is based on the fourth and fifth videos of Lesson 7 (Confidence Intervals).

## 1. Introduction

The construction is done; this notebook makes it operational. First the calculation steps as a checklist you can run on any dataset, then the assumptions that make the checklist valid, then a complete worked example with concrete numbers — the first in this lesson where the interval is an actual pair of centimeters rather than a formula.

After completing this notebook you will be able to:

- Execute the six calculation steps of a confidence interval in order, by hand and in code.
- State the two assumptions behind the recipe and check them for a given scenario.
- Reproduce the Statistopia example: $n = 49$, $\bar{x} = 170$, $\sigma = 25$, 95% → $[163, 177]$.
- Rerun the same data at other confidence levels and predict how the interval changes.
- Validate the recipe end-to-end by simulating repeated samples from a finite population.

## 2. Theory

### The six calculation steps

1. **Find the sample mean** $\bar{x}$.
2. **Define the desired confidence level** $1 - \alpha$ (for example 95%).
3. **Find the critical value** for that level: $z_{1-\alpha/2}$ (from a table or software).
4. **Find the standard error** — the standard deviation of the distribution of sample means:
   $$\text{SE} = \frac{\sigma}{\sqrt{n}}$$
5. **Multiply** critical value × standard error to get the **margin of error**:
   $$\text{margin} = z_{1-\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$
6. **Add and subtract** the margin from the sample mean:
   $$\text{CI} = \left[\bar{x} - \text{margin},\;\; \bar{x} + \text{margin}\right]$$

And there you have it — that's how you calculate a confidence interval.

### The fine print: two assumptions

The recipe is only valid when:

1. **The sample is random.** Every conclusion rests on $\bar{X}$ having the sampling distribution $N(\mu, \sigma^2/n)$; a biased, non-random sample has some *other* distribution and the recipe's promise evaporates — no margin of error can fix how the data was collected.
2. **The sample size exceeds 30, or the population is approximately normal.** This is the central limit theorem's entry fee from the last notebook: a normal population makes $\bar{X}$ exactly normal at any $n$, while a large sample makes it approximately normal for any population.

The lesson's illustrations often used tiny samples ($n = 1, 2, 10$) for clarity — legitimate there because Statistopia's heights were assumed normal. In your own applications, make sure at least one of the two conditions holds.

## 3. Worked Examples

### Example 1: the Statistopia calculation, step by step

Statistopia has 6,000 adults; you want an interval that very likely contains their average height. Measuring all 6,000 is off the table, so you pick $n = 49$ at random and find $\bar{x} = 170$ cm; the population standard deviation is known to be $\sigma = 25$ cm. Target: a 95% confidence interval.

1. sample mean: $\bar{x} = 170$
2. confidence level: $1 - \alpha = 0.95$
3. critical value: $z_{0.975} = 1.96$
4. standard error: $\displaystyle \frac{25}{\sqrt{49}} = \frac{25}{7} \approx 3.57$
5. margin of error: $\displaystyle 1.96 \times \frac{25}{7} = \frac{49}{7} = 7$ cm — conveniently exact, since $1.96 \times 25 = 49$
6. interval: $170 \pm 7 = \boxed{[163,\; 177]}$

That is the interval in which we are 95% confident the average height of Statistopia's adults lies (in the recipe sense the last notebooks made precise).

### Example 2: same data, other levels

Only step 3 changes:

| level | $z_{1-\alpha/2}$ | margin $= z \cdot 25/7$ | interval |
|---|---|---|---|
| 90% | 1.645 | 5.9 cm | $[164.1,\; 175.9]$ |
| 95% | 1.960 | 7.0 cm | $[163.0,\; 177.0]$ |
| 99% | 2.576 | 9.2 cm | $[160.8,\; 179.2]$ |

### Example 3: auditing the assumptions

For the Statistopia example both boxes tick: the 49 were picked **at random** from the 6,000, and $n = 49 > 30$ (heights being roughly normal would have sufficed on its own). A cautionary counter-example: a height survey posted on a basketball forum with $n = 1{,}000$ respondents fails assumption 1 — and no large $n$ repairs a non-random sample.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project), then the six steps as one function that narrates its work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(23)

def z_score(p):
    """Critical value z_p: the point with probability p to its left."""
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

def confidence_interval(x_bar, sigma, n, level=0.95, verbose=False):
    """The six calculation steps, in order."""
    alpha = 1 - level                                   # step 2
    critical = z_score(1 - alpha / 2)                   # step 3
    standard_error = sigma / np.sqrt(n)                 # step 4
    margin = critical * standard_error                  # step 5
    lower, upper = x_bar - margin, x_bar + margin       # step 6
    if verbose:
        print(f"1. sample mean:       x-bar = {x_bar}")
        print(f"2. confidence level:  1 - alpha = {level:.0%}")
        print(f"3. critical value:    z_{1 - alpha / 2} = {critical:.3f}")
        print(f"4. standard error:    {sigma}/sqrt({n}) = {standard_error:.3f}")
        print(f"5. margin of error:   {critical:.3f} * {standard_error:.3f} = {margin:.2f}")
        print(f"6. interval:          {x_bar} +/- {margin:.2f} = [{lower:.1f}, {upper:.1f}]")
    return lower, upper

### The video's example, narrated

In [ ]:
interval_95 = confidence_interval(x_bar=170, sigma=25, n=49, level=0.95, verbose=True)

### Other levels on the same sample

In [ ]:
for level in (0.90, 0.95, 0.99):
    lo, hi = confidence_interval(170, 25, 49, level)
    print(f"{level:.0%}: [{lo:.1f}, {hi:.1f}]   (width {hi - lo:.1f} cm)")

### Validating the recipe on the actual 6,000 islanders

Build Statistopia itself — 6,000 heights — then repeat the survey many times: draw 49 people *without replacement* (as a real survey would), run the six steps, and check how often $[\bar{x} - 7, \bar{x} + 7]$ captures the true island mean.

In [ ]:
N_ISLAND = 6_000
island_heights = rng.normal(170, 25, size=N_ISLAND)
mu_island = island_heights.mean()
print(f"the island's true mean height (normally unknowable): {mu_island:.2f} cm")

n, margin = 49, 7.0
captures = 0
trials = 10_000
for _ in range(trials):
    survey = rng.choice(island_heights, size=n, replace=False)
    x_bar = survey.mean()
    captures += (x_bar - margin <= mu_island <= x_bar + margin)
print(f"surveys whose interval captured the island mean: {captures / trials:.1%} "
      f"(promised: 95%)")

## 5. Visualizations

### The worked example, annotated

Every ingredient of the recipe in one picture.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))

x_bar, margin = 170, 7.0
ax.plot([x_bar - margin, x_bar + margin], [0, 0], color="tab:orange",
        linewidth=8, solid_capstyle="round", alpha=0.85, zorder=1)
ax.plot([x_bar], [0], marker="o", color="tab:blue", markersize=10, zorder=2)

ax.annotate(r"$\bar{x} = 170$", xy=(x_bar, 0.06), ha="center",
            fontsize=11, color="tab:blue")
for x, text in [(163, "163"), (177, "177")]:
    ax.annotate(text, xy=(x, -0.09), ha="center", fontsize=11)
ax.annotate("", xy=(x_bar + margin, -0.035), xytext=(x_bar, -0.035),
            arrowprops=dict(arrowstyle="->", color="black"))
ax.annotate(r"margin $= z_{0.975} \cdot \sigma/\sqrt{n} = 1.96 \times 25/7 = 7$ cm",
            xy=(x_bar + margin / 2, -0.075), ha="center", fontsize=9)
ax.set_xlim(155, 185)
ax.set_ylim(-0.16, 0.14)
ax.set_yticks([])
ax.set_xlabel("height (cm)")
ax.set_title("The 95% confidence interval for Statistopia's mean height: [163, 177]")
plt.tight_layout()
plt.show()

### One sample, three promises

The same survey read at three confidence levels — the nested intervals of Example 2.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.4))
for row, (level, color) in enumerate([(0.90, "tab:green"),
                                      (0.95, "tab:orange"),
                                      (0.99, "tab:red")]):
    lo, hi = confidence_interval(170, 25, 49, level)
    ax.plot([lo, hi], [row, row], color=color, linewidth=7,
            solid_capstyle="round", alpha=0.85)
    ax.annotate(f"{level:.0%}: [{lo:.1f}, {hi:.1f}]",
                xy=(hi + 0.6, row), va="center", fontsize=10)
ax.axvline(170, color="tab:blue", linestyle=":", linewidth=1.5)
ax.annotate(r"$\bar{x}$", xy=(170.4, 2.45), fontsize=11, color="tab:blue")
ax.set_xlim(158, 190)
ax.set_ylim(-0.7, 3.2)
ax.set_yticks([])
ax.set_xlabel("height (cm)")
ax.set_title("Nested promises: more confidence, wider interval -- same sample")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Rerun the Statistopia numbers ($\bar{x} = 170$, $\sigma = 25$, $n = 49$) at the 99.9% level, by hand and with `confidence_interval`.

**Exercise 2.** Recite the six steps in order — then say which single step changes when (a) you demand more confidence, (b) you survey more people, (c) the population is more variable.

### Intermediate

**Exercise 3.** How many islanders must you survey to shrink the 95% margin from 7 cm to at most 2 cm ($\sigma = 25$)? Verify your $n$ with the six-step function.

**Exercise 4.** *Assumption audit.* For each scenario, say whether the recipe applies and which assumption (if any) fails: (a) 12 randomly chosen adults from a normal-height population; (b) 50 randomly sampled household incomes (heavily skewed); (c) 8 randomly sampled incomes; (d) 1,000 self-selected respondents to an online height poll.

### Advanced

**Exercise 5.** *Finite population.* Our simulation sampled 49 of 6,000 without replacement, yet used the with-replacement standard error $\sigma/\sqrt{n}$. Look up the finite population correction $\sqrt{(N - n)/(N - 1)}$, compute it for $N = 6000, n = 49$, and argue numerically why ignoring it was harmless. At what sampling fraction would it start to matter?

**Exercise 6.** *Full validation.* Wrap survey-simulation into a function and check the recipe's coverage on the island at the 90%, 95%, and 99% levels simultaneously. Then break assumption 1 on purpose — survey only islanders taller than 160 cm — and report what happens to the 95% coverage.

## 7. Solutions

In [ ]:
# Exercise 1: the 99.9% interval
z999 = z_score(1 - 0.001 / 2)
m999 = z999 * 25 / 7
print(f"by hand: z = {z999:.3f}, margin = {z999:.3f} * 25/7 = {m999:.2f} cm")
print(f"interval: [{170 - m999:.1f}, {170 + m999:.1f}]")
lo, hi = confidence_interval(170, 25, 49, 0.999)
print(f"function: [{lo:.1f}, {hi:.1f}]")

In [ ]:
# Exercise 2: the steps, and which one moves
print("1 sample mean, 2 confidence level, 3 critical value, 4 standard error,")
print("5 margin = critical x SE, 6 interval = mean +/- margin.")
print("(a) more confidence -> step 3 (bigger critical value)")
print("(b) more people     -> step 4 (smaller standard error)")
print("(c) more variability-> step 4 again (bigger sigma, bigger SE)")

In [ ]:
# Exercise 3: n for a 2 cm margin
z95 = z_score(0.975)
n_needed = int(np.ceil((z95 * 25 / 2) ** 2))
print(f"n >= (1.96 * 25 / 2)^2 = {(z95 * 25 / 2) ** 2:.1f} -> n = {n_needed}")
lo, hi = confidence_interval(170, 25, n_needed, 0.95)
print(f"check at n = {n_needed}: [{lo:.2f}, {hi:.2f}] "
      f"(margin {(hi - lo) / 2:.2f} cm <= 2)")

In [ ]:
# Exercise 4: assumption audit
print("(a) n = 12, random, normal population: VALID -- normality substitutes for n > 30.")
print("(b) n = 50, random, skewed incomes: VALID -- n > 30 invokes the CLT.")
print("(c) n = 8, random, skewed incomes: INVALID -- neither n > 30 nor normality.")
print("(d) n = 1000, self-selected poll: INVALID -- the sample isn't random, and")
print("    no sample size repairs that.")

In [ ]:
# Exercise 5: finite population correction
N, n = 6_000, 49
fpc = np.sqrt((N - n) / (N - 1))
print(f"FPC = sqrt((6000 - 49)/5999) = {fpc:.4f}")
print(f"corrected margin: 7.00 * {fpc:.4f} = {7 * fpc:.3f} cm -- a 0.4% change,")
print("invisible next to the interval's 14 cm width. The correction matters when")
print("n/N stops being tiny:")
for frac in (0.01, 0.1, 0.3, 0.5):
    n_f = int(N * frac)
    print(f"  sampling {frac:.0%} of the island: FPC = "
          f"{np.sqrt((N - n_f) / (N - 1)):.3f}")

In [ ]:
# Exercise 6: multi-level validation, then a broken assumption
def survey_coverage(population, n, level, trials=5_000, biased=False):
    mu_pop = population.mean()
    pool = population[population > 160] if biased else population
    z = z_score(1 - (1 - level) / 2)
    margin = z * 25 / np.sqrt(n)
    hits = 0
    for _ in range(trials):
        x_bar = rng.choice(pool, size=n, replace=False).mean()
        hits += abs(x_bar - mu_pop) <= margin
    return hits / trials

for level in (0.90, 0.95, 0.99):
    print(f"{level:.0%} recipe, random survey: coverage = "
          f"{survey_coverage(island_heights, 49, level):.1%}")

biased_cov = survey_coverage(island_heights, 49, 0.95, biased=True)
print(f"\n95% recipe, surveying only islanders taller than 160 cm: "
      f"coverage = {biased_cov:.1%}")
print("The non-random sample centers x-bar above mu, and the recipe's promise")
print("collapses -- assumption 1 is not optional.")

## 8. Conclusion

- The recipe: **sample mean → confidence level → critical value $z_{1-\alpha/2}$ → standard error $\sigma/\sqrt{n}$ → margin (critical × SE) → interval $\bar{x} \pm$ margin.**
- Statistopia, worked: $n = 49$ of 6,000 adults, $\bar{x} = 170$, $\sigma = 25$, 95% → margin $1.96 \times 25/7 = 7$ cm → $[163, 177]$; the same sample gives $[164.1, 175.9]$ at 90% and $[160.8, 179.2]$ at 99%.
- Two assumptions, both essential: a **random sample** (no sample size repairs a biased one — coverage collapses, as Exercise 6 shows), and **$n > 30$ or an approximately normal population** (the CLT's entry fee).
- Simulated 10,000 real surveys of the island confirm the promise: ~95% of intervals capture the true mean — even sampling without replacement, where the finite population correction (0.996 here) is safely negligible.

One loose end remains from the whole construction: the "magically known" $\sigma$. Up next: what to do when the population standard deviation is unknown too.